# Ferrari Strategy Optimizer — NB5: Deployment
## FastAPI + Streamlit

Kaggle: GPU OFF | Internet ON | Add NB3+NB4 outputs as inputs

In [1]:
# Cell 1 — Install
import subprocess
subprocess.run(["pip","install","fastapi","uvicorn","streamlit","pyngrok","nest-asyncio","--quiet"], check=True)

import os, json, warnings, threading, time, requests
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok

nest_asyncio.apply()
warnings.filterwarnings("ignore")
print("All packages ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.3 MB/s eta 0:00:00
All packages ready


In [2]:
# Cell 2 — Load Models

def find_file(filename):
    for root, dirs, files in os.walk("/kaggle/"):
        for f in files:
            if f == filename:
                path = os.path.join(root, f)
                print("Found:", path)
                return path
    raise FileNotFoundError(filename + " not found")

import xgboost as xgb
xgb_model = joblib.load(find_file('xgb_pit_predictor.pkl'))
with open(find_file('model_meta.json')) as f:
    xgb_config = json.load(f)
FEATURE_COLS = xgb_config["feature_cols"]
print("XGBoost loaded -", len(FEATURE_COLS), "features")

class TyreDegLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, pred_horizon=5, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.attention = nn.Sequential(nn.Linear(hidden_size, 64), nn.Tanh(), nn.Linear(64, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(nn.Linear(hidden_size, 64), nn.ReLU(),
                                nn.Dropout(dropout * 0.5), nn.Linear(64, pred_horizon))
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = (attn_weights * lstm_out).sum(dim=1)
        return self.fc(self.dropout(context))

try:
    ckpt = torch.load(find_file("lstm_full.pt"), map_location="cpu")
    lstm_model = TyreDegLSTM(**ckpt["model_config"])
    lstm_model.load_state_dict(ckpt["model_state_dict"])
    lstm_model.eval()
    LSTM_FEATURES = ckpt["lstm_features"]
    SEQ_LEN = ckpt["seq_len"]
    scaler_X = joblib.load(find_file("lstm_scaler_X.joblib"))
    scaler_y = joblib.load(find_file("lstm_scaler_y.joblib"))
    LSTM_AVAILABLE = True
    print("LSTM loaded -", len(LSTM_FEATURES), "features")
except Exception as e:
    LSTM_AVAILABLE = False
    print("LSTM not found, XGBoost only:", e)

print("All models ready")

Found: /kaggle/input/notebooks/adityasharma100203/f1-nb3/models/xgb_pit_predictor.pkl
Found: /kaggle/input/notebooks/adityasharma100203/f1-nb3/models/model_meta.json
XGBoost loaded - 23 features
Found: /kaggle/input/notebooks/adityasharma100203/f1-nb4/data/lstm_full.pt
LSTM not found, XGBoost only: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please

In [3]:
# Cell 3 — Strategy Simulator

PIT_LOSS = {
    "Australia": 22.4, "China": 21.8, "Japan": 23.1, "Bahrain": 21.5,
    "Saudi Arabia": 22.0, "Miami": 21.9, "Monaco": 23.5, "Canada": 21.6,
    "Britain": 22.1, "Hungary": 22.3, "Monza": 21.2, "Singapore": 23.8,
    "Abu Dhabi": 21.4, "default": 22.0
}
SC_PRIOR = {
    "Australia": 0.85, "Canada": 0.72, "China": 0.62, "Singapore": 0.58,
    "Bahrain": 0.35, "Saudi Arabia": 0.38, "Monaco": 0.45, "Britain": 0.32,
    "Japan": 0.05, "Monza": 0.05, "Hungary": 0.08, "default": 0.20
}
TYPICAL_LIFE = {"SOFT": 18, "MEDIUM": 25, "HARD": 35}

def run_strategy(circuit, current_lap, total_laps, compound,
                 tyre_age, position, gap_ahead, pit_prob, deg_forecast=None):
    pit_loss  = PIT_LOSS.get(circuit, 22.0)
    sc_prob   = SC_PRIOR.get(circuit, 0.20)
    laps_left = total_laps - current_lap
    typical   = TYPICAL_LIFE.get(compound, 25)
    if deg_forecast and len(deg_forecast) > 1:
        deg_rate = float(np.mean(np.diff(deg_forecast)))
    else:
        deg_rate = 0.03 + (tyre_age / typical) * 0.12
    undercut_prob = min(0.95, max(0.05, 0.3 + pit_prob*0.4 + deg_rate*2 - gap_ahead*0.02))
    undercut_gain = gap_ahead*0.4 + deg_rate*6 - pit_loss*0.15
    overcut_prob  = min(0.80, max(0.05, 0.25 + min(5,laps_left-5)*0.04 - deg_rate*1.5 - pit_prob*0.2))
    overcut_gain  = pit_loss*0.12 - deg_rate*min(5,laps_left-5)
    sc_value      = pit_loss * sc_prob
    sc_recommend  = sc_prob > 0.35 and laps_left > 10
    laps_left_tyre = typical - tyre_age
    if pit_prob > 0.65:
        action = "PIT NOW"
        strategy = "UNDERCUT" if undercut_prob > overcut_prob else "REACTIVE"
        detail = f"Box this lap. {strategy} window open - {undercut_prob:.0%} success. Net gain: +{max(0,undercut_gain):.1f}s"
    elif sc_recommend and sc_prob > 0.5:
        action, strategy = "WAIT - SC LIKELY", "SAFETY CAR"
        detail = f"{sc_prob:.0%} SC probability. Free pit worth {pit_loss:.1f}s. Expected: +{sc_value:.1f}s"
    elif pit_prob > 0.35:
        action, strategy = "MONITOR", "WINDOW OPENING"
        detail = f"Window opens in ~{max(1,laps_left_tyre-2):.0f} laps. Deg rate: +{deg_rate:.3f}s/lap"
    else:
        action, strategy = "STAY OUT", "EXTEND"
        detail = f"Tyres good. ~{laps_left_tyre:.0f} laps remaining. Stable at +{deg_rate:.3f}s/lap"
    confidence = pit_prob if "PIT" in action else sc_prob if "SC" in action else pit_prob if "MON" in action else 1-pit_prob
    return {
        "action": action, "strategy": strategy, "confidence": round(confidence,3),
        "detail": detail, "pit_probability": round(pit_prob,3), "deg_rate": round(deg_rate,4),
        "undercut": {"probability": round(undercut_prob,3), "net_gain": round(undercut_gain,2)},
        "overcut":  {"probability": round(overcut_prob,3),  "net_gain": round(overcut_gain,2)},
        "safety_car": {"probability": round(sc_prob,3), "expected_value": round(sc_value,2), "recommend": sc_recommend},
        "pit_loss_seconds": pit_loss
    }

test = run_strategy("Australia", 38, 57, "MEDIUM", 16, 2, 1.2, 0.82)
print("Test:", test["action"], "|", test["strategy"])
print("Simulator ready")

Test: PIT NOW | UNDERCUT
Simulator ready


In [4]:
# Cell 4 — FastAPI App

app = FastAPI(title="Ferrari Strategy Optimizer", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class LapInput(BaseModel):
    circuit: str = "Australia"
    current_lap: int = 38
    total_laps: int = 57
    tyre_compound: str = "MEDIUM"
    tyre_age: int = 16
    position: int = 2
    gap_ahead: float = 1.2
    lap_time_sec: float = 79.5
    deg_rate_3lap: float = 0.08
    laptime_trend: float = 0.05
    lap_history: Optional[List[float]] = None

def build_features(inp):
    compound_map = {"SOFT": 0, "MEDIUM": 1, "HARD": 2}
    typical = TYPICAL_LIFE.get(inp.tyre_compound, 25)
    laps_rem = inp.total_laps - inp.current_lap
    race_prog = inp.current_lap / inp.total_laps
    sc_prior = SC_PRIOR.get(inp.circuit, 0.2)
    enc = compound_map.get(inp.tyre_compound, 1)
    feat = {
        "TyreLife": inp.tyre_age, "compound_encoded": enc,
        "stint_progress_pct": inp.tyre_age/typical, "compound_adj_tyreage": inp.tyre_age/typical,
        "deg_rate_3lap": inp.deg_rate_3lap, "deg_cumulative": inp.deg_rate_3lap*inp.tyre_age,
        "lap_delta_from_fresh": inp.deg_rate_3lap*inp.tyre_age*0.6,
        "deg_x_tyrelife": inp.deg_rate_3lap*inp.tyre_age,
        "LapTimeSec": inp.lap_time_sec, "laptime_fuel_adj": inp.lap_time_sec+laps_rem*0.052,
        "laptime_trend_5lap": inp.laptime_trend, "race_progress_pct": race_prog,
        "laps_remaining": laps_rem, "pit_count_so_far": max(0, inp.tyre_age//typical-1),
        "urgency_score": inp.deg_rate_3lap*race_prog, "Position": inp.position,
        "pos_gap_to_leader": inp.position-1, "position_delta": 0,
        "is_undercut_candidate": int(inp.gap_ahead < 3 and inp.tyre_age > 10),
        "sc_circuit_prior": sc_prior, "sc_opportunity_score": sc_prior*0.7,
        "SafetyCarLap": 0, "team_encoded": 3
    }
    df = pd.DataFrame([feat])
    for col in FEATURE_COLS:
        if col not in df.columns: df[col] = 0
    return df[FEATURE_COLS]

@app.get("/")
def root():
    return {"status": "Ferrari Strategy Optimizer Online",
            "models": {"xgboost": "loaded", "lstm": "loaded" if LSTM_AVAILABLE else "unavailable"}}

@app.get("/health")
def health(): return {"status": "healthy", "lstm": LSTM_AVAILABLE}

@app.post("/predict")
def predict(inp: LapInput):
    pit_prob = float(xgb_model.predict_proba(build_features(inp))[0][1])
    deg_forecast = None
    if LSTM_AVAILABLE and inp.lap_history and len(inp.lap_history) >= SEQ_LEN:
        try:
            seq = np.array(inp.lap_history[-SEQ_LEN:], dtype=np.float32)
            seq_s = scaler_X.transform(seq.reshape(-1, len(LSTM_FEATURES))).reshape(1, SEQ_LEN, -1)
            with torch.no_grad():
                pred = lstm_model(torch.tensor(seq_s)).numpy()
            deg_forecast = scaler_y.inverse_transform(pred)[0].tolist()
        except: pass
    strategy = run_strategy(inp.circuit, inp.current_lap, inp.total_laps,
                            inp.tyre_compound, inp.tyre_age, inp.position,
                            inp.gap_ahead, pit_prob, deg_forecast)
    return {"pit_probability": round(pit_prob,4), "deg_forecast": deg_forecast, "strategy": strategy}

print("FastAPI ready - 3 endpoints defined")

FastAPI ready - 3 endpoints defined


In [5]:
# Cell 5 — Launch API
# Get free token: ngrok.com -> Sign up -> Dashboard -> Your Authtoken

NGROK_TOKEN = "Paste Ngrok Token Here"  # paste your token here
API_PORT = 8000

ngrok.set_auth_token(NGROK_TOKEN)

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="error")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(3)

public_url = ngrok.connect(API_PORT)
API_URL = str(public_url)

print("FastAPI LIVE!")
print("Public URL :", API_URL)
print("Docs URL   :", API_URL + "/docs")

time.sleep(2)
r = requests.get(f"http://localhost:{API_PORT}/")
print("Health check:", r.json()["status"])

FastAPI LIVE!                                                                                       
Public URL : NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8000"
Docs URL   : NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8000"/docs
Health check: Ferrari Strategy Optimizer Online


In [6]:
# Cell 6 — Test API

payload = {
    "circuit": "Australia", "current_lap": 38, "total_laps": 57,
    "tyre_compound": "MEDIUM", "tyre_age": 16, "position": 2,
    "gap_ahead": 1.2, "lap_time_sec": 79.207,
    "deg_rate_3lap": 0.09, "laptime_trend": 0.07
}
r = requests.post(f"http://localhost:{API_PORT}/predict", json=payload)
result = r.json()
print("TEST - Leclerc, Australia, Lap 38")
print("Pit Probability :", result["pit_probability"])
print("Action          :", result["strategy"]["action"])
print("Strategy        :", result["strategy"]["strategy"])
print("Detail          :", result["strategy"]["detail"])
print("API working!")

TEST - Leclerc, Australia, Lap 38
Pit Probability : 0.1145
Action          : WAIT - SC LIKELY
Strategy        : SAFETY CAR
Detail          : 85% SC probability. Free pit worth 22.4s. Expected: +19.0s
API working!


In [7]:
# Cell 7 — Write Streamlit App

app_code = """
import streamlit as st
import requests

API = "http://localhost:8000"

st.set_page_config(page_title="Ferrari Strategy Optimizer", page_icon="F1", layout="wide")

st.markdown(\"\"\"
<style>
.stApp{background:#060404;color:#f2ece8}
.card{background:#110808;border:1px solid rgba(220,0,0,0.3);padding:16px;border-radius:4px;text-align:center;margin:4px}
.val{font-size:28px;font-weight:900;color:#DC0000}
.lbl{font-size:9px;letter-spacing:2px;color:#9a8078}
.rec{background:rgba(220,0,0,0.08);border:1px solid rgba(220,0,0,0.5);padding:20px;border-radius:4px}
h1,h2,h3{color:#f2ece8 !important}
.stButton>button{background:#DC0000 !important;color:white !important;border:none !important;font-weight:700 !important}
</style>
\"\"\", unsafe_allow_html=True)

st.markdown(\"\"\"
<div style="border-bottom:2px solid #DC0000;padding-bottom:12px;margin-bottom:20px">
<div style="font-size:9px;letter-spacing:4px;color:#DC0000">SCUDERIA FERRARI STRATEGY INTELLIGENCE 2026</div>
<div style="font-size:34px;font-weight:900">FORZA STRATEGY ENGINE</div>
<div style="font-size:12px;color:#9a8078;font-style:italic">XGBoost Pit Classifier | LSTM Tyre Forecaster | Monte Carlo Simulator</div>
</div>
\"\"\", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("### Race Parameters")
    circuit = st.selectbox("Circuit",["Australia","China","Japan","Bahrain","Saudi Arabia","Miami","Monaco","Canada","Britain","Hungary","Monza","Singapore","Abu Dhabi"])
    total_laps = st.slider("Total Laps",40,78,57)
    current_lap = st.slider("Current Lap",1,total_laps,38)
    st.markdown("---")
    compound = st.selectbox("Compound",["SOFT","MEDIUM","HARD"])
    tyre_age = st.slider("Tyre Age (laps)",1,50,16)
    st.markdown("---")
    position = st.slider("Position",1,20,2)
    gap_ahead = st.slider("Gap to Ahead (s)",0.0,30.0,1.2,0.1)
    st.markdown("---")
    lap_time = st.number_input("Last Lap (s)",70.0,120.0,79.207,0.001,"%.3f")
    deg_rate = st.slider("Deg Rate",0.0,0.5,0.09,0.01)
    lap_trend = st.slider("Lap Trend",-0.3,0.5,0.07,0.01)
    run = st.button("RUN ANALYSIS",use_container_width=True)

if run:
    with st.spinner("Computing..."):
        payload = {
            "circuit": circuit, "current_lap": current_lap, "total_laps": total_laps,
            "tyre_compound": compound, "tyre_age": tyre_age, "position": position,
            "gap_ahead": gap_ahead, "lap_time_sec": lap_time,
            "deg_rate_3lap": deg_rate, "laptime_trend": lap_trend
        }
        try:
            r = requests.post(f"{API}/predict", json=payload, timeout=10)
            data = r.json()
        except Exception as e:
            st.error(f"API Error: {e}")
            st.stop()

    s = data["strategy"]
    p = data["pit_probability"]
    color = "#DC0000" if "PIT" in s["action"] else "#00FF88" if "STAY" in s["action"] else "#FFD700"

    action_html = s["action"]
    strategy_html = s["strategy"]
    detail_html = s["detail"]
    conf = s["confidence"]
    conf_pct = round(conf * 100, 1)
    conf_bar = round(conf * 100)

    st.markdown(f\"\"\"
    <div class="rec">
    <div style="font-size:9px;letter-spacing:3px;color:#9a8078">STRATEGY RECOMMENDATION</div>
    <div style="font-size:42px;font-weight:900;color:{color}">{action_html}</div>
    <div style="font-size:10px;letter-spacing:2px;color:#9a8078">{strategy_html}</div>
    <div style="font-size:14px;color:#c0a080;margin-top:8px;font-style:italic">{detail_html}</div>
    <div style="margin-top:12px">
    <div style="font-size:8px;color:#9a8078;letter-spacing:2px">CONFIDENCE</div>
    <div style="background:rgba(255,255,255,0.05);height:6px;border-radius:3px;margin-top:4px">
    <div style="width:{conf_bar}%;background:#DC0000;height:6px;border-radius:3px"></div>
    </div>
    <div style="font-size:11px;color:#DC0000;margin-top:3px">{conf_pct}%</div>
    </div></div>
    \"\"\", unsafe_allow_html=True)

    st.markdown("---")
    c1,c2,c3,c4,c5 = st.columns(5)
    pit_pct   = round(p * 100, 1)
    uc_pct    = round(s["undercut"]["probability"] * 100)
    oc_pct    = round(s["overcut"]["probability"] * 100)
    sc_pct    = round(s["safety_car"]["probability"] * 100)
    deg_val   = s["deg_rate"]

    for col, val, label in [
        (c1, f"{pit_pct}%",  "PIT PROB"),
        (c2, f"{uc_pct}%",   "UNDERCUT"),
        (c3, f"{oc_pct}%",   "OVERCUT"),
        (c4, f"{sc_pct}%",   "SC PROB"),
        (c5, f"{deg_val}s",  "DEG/LAP")
    ]:
        col.markdown(
            f'<div class="card"><div class="val">{val}</div><div class="lbl">{label}</div></div>',
            unsafe_allow_html=True
        )

    uc = s["undercut"]; oc = s["overcut"]; sc = s["safety_car"]
    st.markdown("---")
    st.markdown("### Strategy Simulations")
    sc1, sc2, sc3 = st.columns(3)
    with sc1:
        st.markdown(f'<div class="card"><div style="font-size:8px;letter-spacing:2px;color:#DC0000">UNDERCUT</div><div class="val">PIT NOW</div><div class="lbl">SUCCESS {round(uc["probability"]*100)}%</div><div class="lbl">GAIN +{max(0,round(uc["net_gain"],1))}s</div></div>', unsafe_allow_html=True)
    with sc2:
        st.markdown(f'<div class="card"><div style="font-size:8px;letter-spacing:2px;color:#FFD700">OVERCUT</div><div class="val">STAY</div><div class="lbl">SUCCESS {round(oc["probability"]*100)}%</div><div class="lbl">GAIN +{max(0,round(oc["net_gain"],1))}s</div></div>', unsafe_allow_html=True)
    with sc3:
        sc_word = "WAIT" if sc["recommend"] else "UNLIKELY"
        st.markdown(f'<div class="card"><div style="font-size:8px;letter-spacing:2px;color:#4080FF">SAFETY CAR</div><div class="val">{sc_word}</div><div class="lbl">PROB {round(sc["probability"]*100)}%</div><div class="lbl">VALUE {round(sc["expected_value"],1)}s</div></div>', unsafe_allow_html=True)

    st.markdown("<div style='text-align:center;color:#5a3830;font-size:11px;margin-top:20px;font-style:italic'>Ferrari Strategy Optimizer | MTech DS 2026 | La Rossa non si ferma mai</div>", unsafe_allow_html=True)

else:
    st.markdown("<div style='text-align:center;padding:80px 0;color:#5a3830'><div style='font-size:60px'>F1</div><div style='font-size:16px;margin-top:16px;letter-spacing:2px'>SET PARAMETERS IN SIDEBAR</div><div style='font-size:12px;margin-top:8px;font-style:italic'>Then click RUN ANALYSIS</div></div>", unsafe_allow_html=True)
"""

with open("/kaggle/working/streamlit_app.py", "w") as f:
    f.write(app_code)
print("Streamlit app written successfully")

Streamlit app written successfully


In [8]:
# Cell 8 — Launch Streamlit + Get Public URL

STREAMLIT_PORT = 8501
st_thread = threading.Thread(
    target=lambda: os.system(
        f"streamlit run /kaggle/working/streamlit_app.py "
        f"--server.port {STREAMLIT_PORT} --server.headless true "
        f"--server.enableCORS false --server.enableXsrfProtection false "
        f"> /kaggle/working/streamlit.log 2>&1"
    ), daemon=True
)
st_thread.start()
time.sleep(6)

st_tunnel = ngrok.connect(STREAMLIT_PORT)
STREAMLIT_URL = str(st_tunnel)

print("DEPLOYMENT COMPLETE!")
print("=" * 50)
print("Streamlit Dashboard :", STREAMLIT_URL)
print("FastAPI Backend     :", API_URL)
print("API Docs            :", API_URL + "/docs")
print("=" * 50)
print("Forza Ferrari!")

DEPLOYMENT COMPLETE!
Streamlit Dashboard : NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8501"
FastAPI Backend     : NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8000"
API Docs            : NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8000"/docs
Forza Ferrari!


In [ ]:
# Cell 9 — Keep Alive
print("Ferrari Strategy Optimizer - LIVE")
print("Dashboard:", STREAMLIT_URL)
print("Monitoring... interrupt to stop")
i = 0
while True:
    time.sleep(30)
    i += 1
    try:
        r = requests.get(f"http://localhost:{API_PORT}/health", timeout=5)
        status = "ONLINE" if r.status_code == 200 else "DEGRADED"
    except:
        status = "OFFLINE"
    print(f"[{i*30}s] API: {status}")

Ferrari Strategy Optimizer - LIVE
Dashboard: NgrokTunnel: "https://unrivaled-ulysses-unchromed.ngrok-free.dev" -> "http://localhost:8501"
Monitoring... interrupt to stop
[30s] API: ONLINE
[60s] API: ONLINE
[90s] API: ONLINE
[120s] API: ONLINE
[150s] API: ONLINE
